# 09. Финальная оценка Logistic Regression на test

Протокол оценки ранее выбранной модели `LogisticRegression_text_balanced.joblib` на неизменённом CSV. Обучение и подбор параметров здесь отсутствуют. Итоговые метрики и предсказания сохранены в `results/multiclass/final_test/baseline/`. Флаг `RUN_FINAL_TEST=False` по умолчанию блокирует новую оценку; существующий результат защищён от перезаписи.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.metrics import (f1_score,precision_score,recall_score,accuracy_score,confusion_matrix)
import joblib

CLASSES = ['no_risk','cyber_risk','operational_risk','regulatory_risk']
DATA_SHA = '52333a13e7fb6f6c4162486198fbf1d1ab368672a8974295995580a1924f67f3'
name = 'LogisticRegression_text_balanced'
roots=[Path.cwd(),Path.cwd().parent]+[p for p in Path.cwd().iterdir() if p.is_dir()]
ROOT=next((p for p in roots if (p/'data/bank_risk_dataset.csv').is_file() and
    (p/f'results/multiclass/baseline/{name}.joblib').is_file()),None)
if ROOT is None: raise FileNotFoundError('В рабочем каталоге отсутствуют необходимые файлы проекта.')
source = ROOT/'data/bank_risk_dataset.csv'
assert hashlib.sha256(source.read_bytes()).hexdigest()==DATA_SHA
parts=pd.read_csv(source)
assert parts['sample_id'].is_unique
assert len(parts)==5041 and len(parts.query("split == 'test'"))==751
assert set(parts['risk_type_4cls'])==set(CLASSES)
selection=json.loads((ROOT/'results/multiclass/baseline/selection.json').read_text('utf8'))
assert selection['name']==name and selection['dataset_sha256']==DATA_SHA
print('CSV и модель baseline: OK.')

Проверено: исходный CSV, выбранная модель, сохранённые веса baseline.


In [2]:
RUN_FINAL_TEST = False
outdir=ROOT/'results/multiclass/final_test/baseline'
output=outdir/'test_metrics.json'
if not RUN_FINAL_TEST:
    print('Test отключён: RUN_FINAL_TEST=False.')
elif output.exists():
    print('Сохранённый test:', output)
    print(json.loads(output.read_text('utf8')))
else:
    model=joblib.load(ROOT/f'results/multiclass/baseline/{name}.joblib')
    test=parts.loc[parts['split']=='test',['sample_id','risk_type_4cls','text_fragment']].copy()
    texts=test['text_fragment'].fillna('').astype(str).tolist()
    predicted=np.asarray(model.predict(texts))
    if predicted.dtype.kind in 'iu':
        pred_labels=np.asarray(CLASSES)[predicted.astype(int)]
    else:
        pred_labels=predicted.astype(str)
    truth=test['risk_type_4cls'].to_numpy()
    assert set(pred_labels).issubset(set(CLASSES))
    class_f1=f1_score(truth,pred_labels,labels=CLASSES,average=None,zero_division=0)
    metrics={'model':name,'family':'baseline','split':'test','seed':42,'dataset_sha256':DATA_SHA,
       'n_samples':len(test),'macro_f1':float(class_f1.mean()),
       'risk_macro_f1':float(class_f1[1:].mean()),
       'macro_precision':float(precision_score(truth,pred_labels,labels=CLASSES,average='macro',zero_division=0)),
       'macro_recall':float(recall_score(truth,pred_labels,labels=CLASSES,average='macro',zero_division=0)),
       'weighted_f1':float(f1_score(truth,pred_labels,labels=CLASSES,average='weighted',zero_division=0)),
       'accuracy':float(accuracy_score(truth,pred_labels)),
       'confusion_matrix':confusion_matrix(truth,pred_labels,labels=CLASSES).tolist()}
    metrics.update({'f1_'+cls:float(score) for cls,score in zip(CLASSES,class_f1)})
    outdir.mkdir(parents=True,exist_ok=True)
    (outdir/'test_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),'utf8')
    pd.DataFrame({'sample_id':test['sample_id'].to_numpy(),'true':truth,'prediction':pred_labels}).to_csv(outdir/'test_predictions.csv',index=False)
    print('Финальный test сохранён:',output)
    print('Macro F1:',metrics['macro_f1'])

Финальный test сохранён: d:\bank_risk_diploma_updated_2026_10_25\results\multiclass\final_test\baseline\test_metrics.json
Macro F1: 0.8206431656888438


## Сохранённый результат

Итоговый Macro F1 равен 0,8206. Предсказания используются в сравнительном отчёте 11 вместе с результатами mDeBERTa. Настройки модели по результатам test не изменялись. Output относится к завершённой оценке; повторный запуск отключён (`RUN_FINAL_TEST=False`).